In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = 'cpu'
device

'cpu'

In [3]:
class NCFDataset(Dataset):
    def __init__(self, user_ids, item_ids, ratings):
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.ratings[idx]

In [4]:
# GMF Class (Generalized Matrix Factorization)
class GMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super(GMF, self).__init__()
        # Embedding layers for users and items
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)

    def forward(self, user_ids, item_ids):
        user_emb = self.user_embedding(user_ids)  # Shape: [batch_size, latent_dim]
        item_emb = self.item_embedding(item_ids)  # Shape: [batch_size, latent_dim]
        
        # GMF: Element-wise product
        return user_emb * item_emb  # Shape: [batch_size, latent_dim]

In [5]:
# MLP Class (Multi-Layer Perceptron)
class MLP(nn.Module):
    def __init__(self, latent_dim, hidden_layers):
        super(MLP, self).__init__()
        input_dim = latent_dim * 2  # Concatenated user and item embeddings
        layers = []
        for units in hidden_layers:
            layers.append(nn.Linear(input_dim, units))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            input_dim = units
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids, item_ids, user_embedding, item_embedding):
        # Concatenate user and item embeddings
        x = torch.cat([user_embedding, item_embedding], dim=-1)  # Shape: [batch_size, latent_dim * 2]
        return self.mlp(x)  # Shape: [batch_size, last_hidden_layer_size]

In [6]:
# NCF Class (Neural Collaborative Filtering) with GMF and MLP
class NCF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=16, hidden_layers=[64, 32, 16]):
        super(NCF, self).__init__()
        
        # GMF component
        self.gmf = GMF(num_users, num_items, latent_dim)
        
        # MLP component
        self.mlp = MLP(latent_dim, hidden_layers)
        
        # Output Layer (Fusion of GMF and MLP)
        fusion_dim = latent_dim + hidden_layers[-1]  # GMF (latent_dim) + MLP (last hidden layer)
        self.output_layer = nn.Linear(fusion_dim, 1)  # Single output value for regression
        
    def forward(self, user_ids, item_ids):
        # Get GMF output (element-wise product of embeddings)
        gmf_output = self.gmf(user_ids, item_ids)
        
        # Get MLP output (processed through hidden layers)
        user_emb = self.gmf.user_embedding(user_ids)
        item_emb = self.gmf.item_embedding(item_ids)
        mlp_output = self.mlp(user_ids, item_ids, user_emb, item_emb)
        
        # Combine GMF and MLP outputs
        combined = torch.cat([gmf_output, mlp_output], dim=-1)  # Shape: [batch_size, fusion_dim]
        
        # Final prediction (single value for regression)
        output = self.output_layer(combined)  # Shape: [batch_size, 1]
        return output

In [7]:
attraction_df = pd.read_csv('../Data/FinalDataset/attractions_final.csv')
ratings_df = pd.read_csv('../Data/FinalDataset/ratings.csv')
ratings_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4023 entries, 0 to 4022
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   user_id   4023 non-null   int64  
 1   place_id  2843 non-null   float64
 2   rating    4023 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 94.4 KB


In [8]:
ratings_df['place_id'] = ratings_df['place_id'].fillna(-1)
ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'place_id'])



In [9]:
# Initialize label encoders for user_id and place_id
user_encoder = LabelEncoder()
place_encoder = LabelEncoder()

# Fit and transform the user_id and place_id to continuous integers starting from 0 
ratings_df['user_id'] = user_encoder.fit_transform(ratings_df['user_id'])
ratings_df['place_id'] = place_encoder.fit_transform(ratings_df['place_id'])

# Now the user_id and place_id are mapped to indices starting from 0
num_users = len(user_encoder.classes_)  # Number of unique users
num_items = len(place_encoder.classes_)  # Number of unique places
print(num_users, num_items)

3251 454


In [10]:
# Convert user_id, place_id, and ratings to tensors
user_ids = torch.tensor(ratings_df['user_id'].values, dtype=torch.long)
item_ids = torch.tensor(ratings_df['place_id'].values, dtype=torch.long)
ratings = torch.tensor(ratings_df['rating'].values, dtype=torch.float32)
print(user_ids.max())  # This should be < num_users
print(item_ids.max())  # This should be < num_items


tensor(3250)
tensor(453)


In [11]:
# Create dataset and dataloader
dataset = NCFDataset(user_ids, item_ids, ratings)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [12]:
# Initialize the model, loss function, and optimizer
model = NCF(num_users, num_items, latent_dim=16, hidden_layers=[64, 32, 16]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [13]:
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for user_ids, item_ids, ratings in train_loader:
        user_ids, item_ids, ratings = user_ids.to(device), item_ids.to(device), ratings.to(device)
        optimizer.zero_grad()
        predictions = model(user_ids, item_ids)
        loss = criterion(predictions.squeeze(), ratings)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 15.3862
Epoch 2/10, Loss: 2.8945
Epoch 3/10, Loss: 2.2840
Epoch 4/10, Loss: 2.1678
Epoch 5/10, Loss: 2.0390
Epoch 6/10, Loss: 1.9297
Epoch 7/10, Loss: 1.9105
Epoch 8/10, Loss: 1.8799
Epoch 9/10, Loss: 1.7543
Epoch 10/10, Loss: 1.7561


In [14]:
# Test: Making predictions
model.eval()
with torch.no_grad():
    test_user_ids = torch.tensor([0, 1, 2, 3]).to(device)  # Example test users
    test_item_ids = torch.tensor([10, 11, 12, 13]).to(device)  # Example test items
    predicted_ratings = model(test_user_ids, test_item_ids)
    print(predicted_ratings.squeeze())  # Predicted ratings for test users and itemss

tensor([3.4590, 3.7911, 4.3103, 3.8744])
